In [1]:
import pandas as pd
import glob
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.model_selection import KFold
import os

In [2]:
all_files = glob.glob("data/sold/raw/*.csv")

df = pd.concat(
    (pd.read_csv(f, low_memory=False).assign(source_file=f) for f in all_files),
    ignore_index=True
)

target = 'ClosePrice'

restricted_df = df[
    (df['PropertyType'] == 'Residential') &
    (df['PropertySubType'] == 'SingleFamilyResidence')
].copy()

In [3]:
# Handling Missing Target Values and Duplicate Values

before = len(restricted_df)

# Drop rows where the target is missing or zero
clean_df = restricted_df[restricted_df[target] > 0]
after_target = len(clean_df)

# Drop duplicate rows
cols_no_src = [c for c in clean_df.columns if c != 'source_file']
clean_df = clean_df.drop_duplicates(subset=cols_no_src)
after_dupes = len(clean_df)

print(f'Start:                  {before:,}')
print(f'Dropped bad target:     {before - after_target}')
print(f'Dropped duplicates:     {after_target - after_dupes}')
print(f'Final rows:             {after_dupes:,}')

clean_df = clean_df.reset_index(drop=True)

Start:                  358,332
Dropped bad target:     3
Dropped duplicates:     31
Final rows:             358,298


In [4]:
# Drop Rows with Close Dates before Listing Dates

clean_df['ListingContractDate'] = pd.to_datetime(clean_df['ListingContractDate'])
bad = clean_df['CloseDate'] < clean_df['ListingContractDate']
print(f'CloseDate before listing date: {bad.sum()} rows')

clean_df = clean_df[~bad].reset_index(drop=True)

CloseDate before listing date: 51 rows


In [5]:
# Identifiers
ID_COLS = ['ListingKey', 'ListingId', 'ListingKeyNumeric', 'StreetNumberNumeric', 'UnparsedAddress', 'source_file']

NEVER_USE = [
    'ClosePrice',                                              # target
    'CloseDate', 'PurchaseContractDate',                       # listing/sale dates
    'ContractStatusChangeDate', 'DaysOnMarket', 'close_month',
    'MlsStatus', 'PropertyType', 'PropertySubType',            # constant after filtering
]

# Fields that only exist once a listing/sale exists
LISTING_KEYWORDS = ('list', 'buyer')

def is_listing_col(c):
    return any(k in c.lower() for k in LISTING_KEYWORDS)

NEVER_USE += [c for c in clean_df.columns
              if is_listing_col(c)
                and c not in ID_COLS
                and c not in NEVER_USE]

print('Information that’s not available for any address at the time of prediction, constant, or identifiers:\n')
print(*sorted(NEVER_USE), sep='\n')

Information that’s not available for any address at the time of prediction, constant, or identifiers:

BuyerAgencyCompensation
BuyerAgencyCompensationType
BuyerAgentAOR
BuyerAgentFirstName
BuyerAgentLastName
BuyerAgentMlsId
BuyerOfficeAOR
BuyerOfficeName
CloseDate
ClosePrice
CoBuyerAgentFirstName
CoListAgentFirstName
CoListAgentLastName
CoListOfficeName
ContractStatusChangeDate
DaysOnMarket
ListAgentAOR
ListAgentEmail
ListAgentFirstName
ListAgentFullName
ListAgentLastName
ListOfficeName
ListPrice
ListingContractDate
MlsStatus
OriginalListPrice
PropertySubType
PropertyType
PurchaseContractDate
close_month


In [6]:
# Missing Values

allowed = [c for c in clean_df.columns
                if c not in set(NEVER_USE + ID_COLS)]

missing = clean_df[allowed].isna().mean().mul(100).round(1).sort_values(ascending=False)
print(missing.to_string())

FireplacesTotal                 100.0
WaterfrontYN                    100.0
TaxAnnualAmount                 100.0
AboveGradeFinishedArea          100.0
CoveredSpaces                   100.0
MiddleOrJuniorSchoolDistrict    100.0
ElementarySchoolDistrict        100.0
BusinessType                    100.0
TaxYear                         100.0
BelowGradeFinishedArea           99.4
BasementYN                       97.6
BuilderName                      95.4
LotSizeDimensions                93.9
BuildingAreaTotal                93.6
OriginatingSystemSubName         89.4
OriginatingSystemName            89.4
ElementarySchool                 86.4
MiddleOrJuniorSchool             86.2
HighSchool                       82.2
lonfilled                        74.8
AssociationFeeFrequency          74.8
latfilled                        74.8
SubdivisionName                  65.2
MainLevelBedrooms                40.2
Flooring                         35.7
AssociationFee                   30.1
HighSchoolDi

In [7]:
# Dropped Columns with more than 90% missing values

high_missing = missing[missing > 90]

drop_cols = high_missing.index.tolist()
blocked = set(NEVER_USE + ID_COLS + drop_cols)

remaining = [c for c in clean_df.columns if c not in blocked]

print(f'{len(remaining)} columns remaining\n')
print(*remaining, sep='\n')

36 columns remaining

Flooring
ViewYN
PoolPrivateYN
Latitude
Longitude
LivingArea
AssociationFeeFrequency
MLSAreaMajor
CountyOrParish
ElementarySchool
AttachedGarageYN
ParkingTotal
LotSizeAcres
SubdivisionName
YearBuilt
BathroomsTotalInteger
City
BedroomsTotal
StateOrProvince
MiddleOrJuniorSchool
FireplaceYN
Stories
HighSchool
Levels
LotSizeArea
MainLevelBedrooms
NewConstructionYN
GarageSpaces
HighSchoolDistrict
PostalCode
AssociationFee
LotSizeSquareFeet
latfilled
lonfilled
OriginatingSystemName
OriginatingSystemSubName


In [8]:
def column_types(df, exclude=()):
    rows = []
    for c in df.columns:
        if c in set(exclude):
            continue
        s = df[c]
        n_unique = s.nunique(dropna=True)
        is_text = str(s.dtype) in ['object', 'str', 'string', 'category', 'bool']
        rows.append({
            'col': c,
            'dtype': str(s.dtype),
            'n_unique': n_unique,
            'sample': s.dropna().unique()[:3].tolist(),
            'looks_categorical': is_text,
        })
    return pd.DataFrame(rows).set_index('col')

types = column_types(clean_df, exclude=NEVER_USE + ID_COLS + drop_cols)

# Categorical columns
print('CATEGORICAL COLUMNS\n')
print(types.loc[types['looks_categorical'], ['dtype', 'n_unique', 'sample']].sort_index().to_string())

CATEGORICAL COLUMNS

                           dtype  n_unique                                                                  sample
col                                                                                                               
AssociationFeeFrequency      str         4                                          [Monthly, Annually, Quarterly]
AttachedGarageYN          object         2                                                           [True, False]
City                         str      1132                            [National City, San Luis Obispo, Fort Bragg]
CountyOrParish               str        63                                 [San Diego, San Luis Obispo, Mendocino]
ElementarySchool             str      1381                                    [Other, Oso Grande, Truman Benedict]
FireplaceYN               object         2                                                           [False, True]
Flooring                     str       332                 

In [9]:
# Drop columns only in '_filled' CSVs
drop_cols = drop_cols + ['latfilled', 'lonfilled']

# Drop columns that don't describe the property
drop_cols = drop_cols + ['OriginatingSystemName', 'OriginatingSystemSubName']

In [10]:
# Remove non-California addresses

before = len(clean_df)
is_ca = clean_df['StateOrProvince'].astype('string').str.strip().str.upper().eq('CA')
clean_df = clean_df[is_ca].reset_index(drop=True)
print(f'Kept {len(clean_df):,} CA rows, removed {before - len(clean_df):,} out-of-state rows')

# Constant after the filter, so exclude it from features
drop_cols = drop_cols + ['StateOrProvince']

Kept 358,222 CA rows, removed 25 out-of-state rows


In [11]:
types = column_types(clean_df, exclude=NEVER_USE + ID_COLS + drop_cols)

# Categorical columns
print('CATEGORICAL COLUMNS\n')
print(types.loc[types['looks_categorical'], ['dtype', 'n_unique', 'sample']].sort_index().to_string())

CATEGORICAL COLUMNS

                          dtype  n_unique                                                                  sample
col                                                                                                              
AssociationFeeFrequency     str         4                                          [Monthly, Annually, Quarterly]
AttachedGarageYN         object         2                                                           [True, False]
City                        str      1127                            [National City, San Luis Obispo, Fort Bragg]
CountyOrParish              str        63                                 [San Diego, San Luis Obispo, Mendocino]
ElementarySchool            str      1381                                    [Other, Oso Grande, Truman Benedict]
FireplaceYN              object         2                                                           [False, True]
Flooring                    str       332                          

In [12]:
# Numeric columns
def five_number_summary(df, cols):
    cols = [c for c in cols if c in df.columns]      # skip columns that were dropped
    s = df[cols].describe().T[['min', '25%', '50%', '75%', 'max']]
    s.columns = ['min', 'Q1', 'median', 'Q3', 'max']
    return s.sort_index()

def print_summary(df, cols, title):
    print(f'\n\n{title}\n')
    print(five_number_summary(df, cols).to_string(
        float_format=lambda x: f'{x:,.2f}', col_space=18, justify='right'
    ))

num_cols = types.index[~types['looks_categorical']].tolist()
print_summary(clean_df, num_cols, 'NUMERIC COLUMNS: five-number summary (before cleaning)')



NUMERIC COLUMNS: five-number summary (before cleaning)

                                     min                 Q1             median                 Q3                max
AssociationFee                      0.00               0.00               0.00             130.00          57,500.00
BathroomsTotalInteger               0.00               2.00               2.00               3.00             175.00
BedroomsTotal                       0.00               3.00               3.00               4.00              45.00
GarageSpaces                        0.00               2.00               2.00               2.00             600.00
Latitude                         -117.47              33.77              34.09              35.16              81.00
LivingArea                          0.00           1,380.00           1,808.00           2,428.00         123,764.00
Longitude                        -177.65            -119.64            -118.05            -117.27             329.00
LotSiz

In [13]:
print(clean_df['AssociationFeeFrequency'].value_counts(dropna=False))

AssociationFeeFrequency
NaN             267875
Monthly          79158
Annually          7311
Quarterly         3387
SemiAnnually       491
Name: count, dtype: int64


In [14]:
cols_zero_min = ['BedroomsTotal', 'BathroomsTotalInteger', 'LivingArea']

for c in cols_zero_min:
    s = clean_df[c]
    print(f'{c:24} zeros: {(s == 0).sum():>6,} ({(s == 0).mean():.2%})')

# Set zero values to NaN
for c in cols_zero_min:
    n = (clean_df[c] == 0).sum()
    clean_df.loc[clean_df[c] == 0, c] = np.nan

print_summary(clean_df, cols_zero_min, 'BedroomsTotal, BathroomsTotalInteger, and LivingArea after cleaning')

BedroomsTotal            zeros:    202 (0.06%)
BathroomsTotalInteger    zeros:    156 (0.04%)
LivingArea               zeros:    140 (0.04%)


BedroomsTotal, BathroomsTotalInteger, and LivingArea after cleaning

                                     min                 Q1             median                 Q3                max
BathroomsTotalInteger               1.00               2.00               2.00               3.00             175.00
BedroomsTotal                       1.00               3.00               3.00               4.00              45.00
LivingArea                          2.00           1,380.00           1,809.00           2,428.00         123,764.00


In [15]:
cols_zero_min = cols_zero_min + ['GarageSpaces']

for c in cols_zero_min:
    print(f'\n===== {c} =====')
    print(clean_df[c].quantile([.99, .999, .9999]).to_string())

    cols = list(dict.fromkeys([c, target, 'LivingArea']))  # removes the duplicate


===== BedroomsTotal =====
0.9900     6.0
0.9990     8.0
0.9999    12.0

===== BathroomsTotalInteger =====
0.9900     6.0
0.9990    10.0
0.9999    15.0

===== LivingArea =====
0.9900     5668.14
0.9990    10200.00
0.9999    16489.00

===== GarageSpaces =====
0.9900     4.0000
0.9990     9.0000
0.9999    31.5108


In [16]:
caps = {
    'BedroomsTotal':         12,
    'BathroomsTotalInteger': 15,
    'LivingArea':            16_000,
    'GarageSpaces':          32,
}

for c, cap in caps.items():
    too_big = clean_df[c] > cap
    print(f'{c:24} {too_big.sum():>4,} above {cap:,} set to NaN')
    clean_df.loc[too_big, c] = np.nan

print_summary(clean_df, list(caps), 'After capping')

BedroomsTotal              24 above 12 set to NaN
BathroomsTotalInteger      34 above 15 set to NaN
LivingArea                 41 above 16,000 set to NaN
GarageSpaces               34 above 32 set to NaN


After capping

                                     min                 Q1             median                 Q3                max
BathroomsTotalInteger               1.00               2.00               2.00               3.00              15.00
BedroomsTotal                       1.00               3.00               3.00               4.00              12.00
GarageSpaces                        0.00               2.00               2.00               2.00              32.00
LivingArea                          2.00           1,380.00           1,809.00           2,428.00          15,803.00


In [17]:
# 150 sq ft is the Minimum ADU Size in California
tiny = clean_df['LivingArea'] < 150

print(f'{tiny.sum():,} rows with LivingArea under 150 sq ft')

clean_df = clean_df[~tiny].reset_index(drop=True)

12 rows with LivingArea under 150 sq ft


In [18]:
print_summary(clean_df, ['LivingArea'], 'After minimum')



After minimum

                                  min                 Q1             median                 Q3                max
LivingArea                     192.00           1,380.00           1,809.00           2,428.00          15,803.00


In [19]:
# California
LAT_MIN, LAT_MAX = 32.5, 42.0
LON_MIN, LON_MAX = -124.5, -114.0

lat, lon = clean_df['Latitude'], clean_df['Longitude']

def in_ca(lat, lon):
    return lat.between(LAT_MIN, LAT_MAX) & lon.between(LON_MIN, LON_MAX)

valid_before = in_ca(lat, lon)
print(f'Already valid:        {valid_before.sum():,}')
print(f'Outside CA:           {(~valid_before & lat.notna() & lon.notna()).sum():,}')

# Checked if swapped lat/lon put it in California
swapped = ~valid_before & in_ca(lon, lat)
clean_df.loc[swapped, ['Latitude', 'Longitude']] = (
    clean_df.loc[swapped, ['Longitude', 'Latitude']].values
)
print(f'Fixed by swapping:    {swapped.sum():,}')

# Checked if longitude was missing its minus sign and if it would put it in California
lat, lon = clean_df['Latitude'], clean_df['Longitude']
sign_bug = ~in_ca(lat, lon) & in_ca(lat, -lon)
clean_df.loc[sign_bug, 'Longitude'] = -clean_df.loc[sign_bug, 'Longitude']
print(f'Fixed by sign flip:   {sign_bug.sum():,}')

# 3) Anything still outside California -> NaN (both columns, since a lone coordinate is useless)
lat, lon = clean_df['Latitude'], clean_df['Longitude']
bad = lat.notna() & lon.notna() & ~in_ca(lat, lon)
clean_df.loc[bad, ['Latitude', 'Longitude']] = np.nan
print(f'Set to NaN:           {bad.sum():,}')

print_summary(clean_df, ['Latitude', 'Longitude'], 'Latitude/Longitude after cleaning')

Already valid:        357,743
Outside CA:           81
Fixed by swapping:    0
Fixed by sign flip:   23
Set to NaN:           58


Latitude/Longitude after cleaning

                                  min                 Q1             median                 Q3                max
Latitude                        32.54              33.77              34.09              35.16              41.97
Longitude                     -124.19            -119.64            -118.05            -117.27            -114.31


In [20]:
# Set impossible negative ParkingTotal values to NaN

neg = clean_df['ParkingTotal'] < 0
print(f'Setting {neg.sum()} negative ParkingTotal values to NaN')
clean_df.loc[neg, 'ParkingTotal'] = np.nan

print_summary(clean_df, ['ParkingTotal'], 'ParkingTotal after cleaning')

Setting 82 negative ParkingTotal values to NaN


ParkingTotal after cleaning

                                  min                 Q1             median                 Q3                max
ParkingTotal                     0.00               2.00               2.00               3.00          15,720.00


In [21]:
print(f'\n===== ParkingTotal =====')
print(clean_df['ParkingTotal'].quantile([.99, .999, .9999]).to_string())


===== ParkingTotal =====
0.9900     11.0000
0.9990     25.6105
0.9999    576.2210


In [22]:
col = 'ParkingTotal'
parking_cap = 25

too_big = clean_df[col] > parking_cap
print(f'{col:24} {too_big.sum():>4,} above {parking_cap:,} set to NaN')
clean_df.loc[too_big, col] = np.nan

print_summary(clean_df, [col], 'After capping')

ParkingTotal              359 above 25 set to NaN


After capping

                                  min                 Q1             median                 Q3                max
ParkingTotal                     0.00               2.00               2.00               3.00              25.00


In [23]:
# Compare Lot columns

lot_cols = ['LotSizeAcres', 'LotSizeArea', 'LotSizeSquareFeet']
for c in lot_cols:
    s = clean_df[c]
    print(f'{c:20} zero: {(s == 0).sum():>7,} | missing: {s.isna().sum():>7,} | nonzero: {(s > 0).sum():>7,}')

ok = clean_df[lot_cols].gt(0).all(axis=1) # True where all 3 lot columns are positive
print()
print('LotSizeArea == SqFt:   ', (clean_df.loc[ok, 'LotSizeArea'] == clean_df.loc[ok, 'LotSizeSquareFeet']).mean().round(3))
print('LotSizeArea == Acres:  ', (clean_df.loc[ok, 'LotSizeArea'] == clean_df.loc[ok, 'LotSizeAcres']).mean().round(3))

LotSizeAcres         zero:     543 | missing:   6,155 | nonzero: 351,512
LotSizeArea          zero:     191 | missing:   6,102 | nonzero: 351,917
LotSizeSquareFeet    zero:     190 | missing:   6,125 | nonzero: 351,895

LotSizeArea == SqFt:    0.97
LotSizeArea == Acres:   0.03


In [24]:
# Choose Lot Size SqFt

sqft = 'LotSizeSquareFeet'

# Set zeroes to NaN
zeros = clean_df[sqft] <= 0
clean_df.loc[zeros, sqft] = np.nan
print(f'Zeros set to NaN:        {zeros.sum():,}')

# Fill missing square feet from acres
fill = clean_df[sqft].isna() & (clean_df['LotSizeAcres'] > 0)
clean_df.loc[fill, sqft] = clean_df.loc[fill, 'LotSizeAcres'] * 43560
print(f'Filled from acres:       {fill.sum():,}')

# Drop the other two lot columns
drop_cols = list(dict.fromkeys(drop_cols + ['LotSizeArea', 'LotSizeAcres']))
clean_df = clean_df.drop(columns=drop_cols, errors='ignore')

print(f'{sqft} missing: {clean_df[sqft].isna().mean():.1%}')

print_summary(clean_df, ['LotSizeSquareFeet'], 'LotSizeSquareFeet after cleaning')

Zeros set to NaN:        190
Filled from acres:       19
LotSizeSquareFeet missing: 1.8%


LotSizeSquareFeet after cleaning

                                  min                 Q1             median                 Q3                max
LotSizeSquareFeet                0.01           5,663.00           7,253.00          10,392.00 340,234,020,404.78


In [25]:
sqft = clean_df['LotSizeSquareFeet']

q = sqft.quantile([0.001, 0.01, 0.5, 0.99, 0.999])
print(q.to_string(float_format=lambda x: f'{x:,.2f}'))

0.001           0.59
0.010       1,815.00
0.500       7,253.00
0.990     254,806.63
0.999   3,567,147.57


In [26]:
for t in [10, 100, 250, 500, 750, 1_000, 1_500]:
    print(f'below {t:>5,}: {(sqft < t).sum():>5,} rows')

below    10:   430 rows
below   100:   439 rows
below   250:   456 rows
below   500:   483 rows
below   750:   526 rows
below 1,000:   746 rows
below 1,500: 2,012 rows


In [27]:
# Maximum: 100 acres, which is close to the 99.9% percentile
LOT_MIN, LOT_MAX = 500, 4_356_000

bad_lo = sqft < LOT_MIN
bad_hi = sqft > LOT_MAX
print(f'{bad_lo.sum():,} below {LOT_MIN} and {bad_hi.sum():,} above {LOT_MAX:,} set to NaN')
clean_df.loc[bad_lo | bad_hi, 'LotSizeSquareFeet'] = np.nan

483 below 500 and 319 above 4,356,000 set to NaN


In [28]:
print_summary(clean_df, ['LotSizeSquareFeet'], 'LotSizeSquareFeet after capping')



LotSizeSquareFeet after capping

                                  min                 Q1             median                 Q3                max
LotSizeSquareFeet              500.00           5,663.00           7,254.00          10,374.00       4,356,000.00


In [29]:
mlb, bt = clean_df['MainLevelBedrooms'], clean_df['BedroomsTotal']
bad = mlb > bt

print(f'Impossible rows: {bad.sum():,} ({bad.mean():.2%} of all rows)')
clean_df.loc[bad, 'MainLevelBedrooms'] = np.nan

print_summary(clean_df, ['MainLevelBedrooms', 'BedroomsTotal'], 'MainLevelBedrooms & BedroomsTotal after cleaning')

Impossible rows: 2,143 (0.60% of all rows)


MainLevelBedrooms & BedroomsTotal after cleaning

                                  min                 Q1             median                 Q3                max
BedroomsTotal                    1.00               3.00               3.00               4.00              12.00
MainLevelBedrooms                0.00               1.00               3.00               3.00              18.00


In [30]:
too_big = clean_df['MainLevelBedrooms'] > 12
print(f'{too_big.sum()} MainLevelBedrooms above 12 set to NaN')
clean_df.loc[too_big, 'MainLevelBedrooms'] = np.nan

2 MainLevelBedrooms above 12 set to NaN


In [31]:
print_summary(clean_df, num_cols, 'NUMERIC COLUMNS: five-number summary (after cleaning)')



NUMERIC COLUMNS: five-number summary (after cleaning)

                                     min                 Q1             median                 Q3                max
AssociationFee                      0.00               0.00               0.00             130.00          57,500.00
BathroomsTotalInteger               1.00               2.00               2.00               3.00              15.00
BedroomsTotal                       1.00               3.00               3.00               4.00              12.00
GarageSpaces                        0.00               2.00               2.00               2.00              32.00
Latitude                           32.54              33.77              34.09              35.16              41.97
LivingArea                        192.00           1,380.00           1,809.00           2,428.00          15,803.00
Longitude                        -124.19            -119.64            -118.05            -117.27            -114.31
LotSize

In [32]:
print(clean_df[target].quantile([.001, .01, .5, .99, .999, .9999]))
print((clean_df[target] < 10_000).sum(), 'rows under $10k')

0.0010    9.300000e+04
0.0100    2.370000e+05
0.5000    8.980000e+05
0.9900    6.299910e+06
0.9990    1.728955e+07
0.9999    5.031342e+07
Name: ClosePrice, dtype: float64
8 rows under $10k


In [33]:
before = len(clean_df)

price_min = 10_000
price_max = 50_000_000

bad = (
    (clean_df[target] < price_min)
    | (clean_df[target] > price_max)
)

print(f'Under ${price_min:,}:        {(clean_df[target] < price_min).sum()}')
print(f'Over ${price_max:,}:     {(clean_df[target] > price_max).sum()}')

clean_df = clean_df[~bad].reset_index(drop=True)
print(f'\nDropped {before - len(clean_df)} rows total')

Under $10,000:        8
Over $50,000,000:     36

Dropped 44 rows total


In [34]:
types = column_types(clean_df, exclude=NEVER_USE + ID_COLS + drop_cols)

# Categorical columns
print('CATEGORICAL COLUMNS\n')
print(types.loc[types['looks_categorical'], ['dtype', 'n_unique', 'sample']].sort_index().to_string())

CATEGORICAL COLUMNS

                          dtype  n_unique                                                                  sample
col                                                                                                              
AssociationFeeFrequency     str         4                                          [Monthly, Annually, Quarterly]
AttachedGarageYN         object         2                                                           [True, False]
City                        str      1127                            [National City, San Luis Obispo, Fort Bragg]
CountyOrParish              str        63                                 [San Diego, San Luis Obispo, Mendocino]
ElementarySchool            str      1381                                    [Other, Oso Grande, Truman Benedict]
FireplaceYN              object         2                                                           [False, True]
Flooring                    str       332                          

In [35]:
cat_cols = [c for c in types.index[types['looks_categorical']]
            if c in clean_df.columns and c not in set(NEVER_USE + ID_COLS)]

def pct_blank(s):
    text = s.astype('string').fillna('').str.strip().str.lower()
    return (s.isna() | text.isin(['missing', ''])).mean() * 100

cat_missing = pd.Series({c: pct_blank(clean_df[c]) for c in cat_cols}).round(1).sort_values(ascending=False)

print(f"{'column':26} {'missing %':>9}")
print('-' * 37)
for c, pct in cat_missing.items():
    print(f'{c:26} {pct:>8.1f}%')

column                     missing %
-------------------------------------
ElementarySchool               86.4%
MiddleOrJuniorSchool           86.2%
HighSchool                     82.2%
AssociationFeeFrequency        74.8%
SubdivisionName                65.2%
Flooring                       35.7%
HighSchoolDistrict             26.3%
MLSAreaMajor                   14.4%
AttachedGarageYN               11.7%
PoolPrivateYN                   9.4%
Levels                          9.2%
ViewYN                          9.1%
NewConstructionYN               7.3%
FireplaceYN                     0.1%
City                            0.1%
CountyOrParish                  0.0%
PostalCode                      0.0%


In [36]:
# Categorical columns with a lot of missing values
SPARSE_DROP = ['ElementarySchool', 'MiddleOrJuniorSchool', 'HighSchool', 'SubdivisionName']

# Type of encoding each categorical column gets
binary_cols = ['AttachedGarageYN', 'FireplaceYN', 'NewConstructionYN', 'PoolPrivateYN', 'ViewYN']
te_cols     = ['City', 'CountyOrParish', 'HighSchoolDistrict', 'MLSAreaMajor', 'PostalCode']   # encoding later in the training window

LEVEL_TOKENS = ['one', 'two', 'threeormore', 'multisplit']
FLOOR_TYPES = ['tile', 'carpet', 'wood', 'laminate', 'vinyl', 'stone', 'concrete']
HOA_PER_YEAR = {'monthly': 12, 'quarterly': 4, 'semiannually': 2, 'annually': 1}

YEAR_MIN, HOA_MONTHLY_MAX = 1790, 5_000


def fixed_cleaning(df):
    out, log = df.copy(), {}

    # 1. yes/no -> 1/0
    for c in binary_cols:
        out[c] = out[c].map({True: 1, False: 0})

    # 2. text columns
    out['PostalCode'] = out['PostalCode'].astype('string').str.strip().str[:5]
    out['MLSAreaMajor'] = out['MLSAreaMajor'].astype('string')
    junk = out['MLSAreaMajor'].str.contains('not defined', case=False, na=False)
    out.loc[junk, 'MLSAreaMajor'] = pd.NA
    log['MLSAreaMajor junk -> NaN'] = int(junk.sum())

    out['HighSchoolDistrict'] = out['HighSchoolDistrict'].astype('string').str.strip()
    other = out['HighSchoolDistrict'].str.lower().eq('other').fillna(False)
    out.loc[other, 'HighSchoolDistrict'] = pd.NA
    log["HighSchoolDistrict 'Other' -> NaN"] = int(other.sum())

    for c in ['Levels'] + te_cols:
        out[c] = out[c].astype('string').str.strip().fillna('missing').astype(str)

    # 3. Levels: fixed one-hot
    lv = out['Levels'].astype('string').str.lower()
    for t in LEVEL_TOKENS:
        out[f'levels_{t}'] = lv.str.contains(rf'\b{t}\b', na=False).astype(int)
    level_cols = [f'levels_{t}' for t in LEVEL_TOKENS]
    out['levels_unknown'] = (out[level_cols].sum(axis=1) == 0).astype(int)
    log['rows with more than one level value'] = int((out[level_cols].sum(axis=1) > 1).sum())

    # 4. Flooring: one flag per material plus unknown
    fl = out['Flooring'].astype('string').str.lower().str.strip()
    for m in FLOOR_TYPES:
        out[f'floor_{m}'] = fl.str.contains(m, na=False).astype(int)
    out['floor_unknown'] = (fl.isna() | fl.isin(['missing', 'seeremarks'])).astype(int)

    # 5. HOA: monthly amount and missingness flag
    fee = pd.to_numeric(out['AssociationFee'], errors='coerce')

    mult = (
        out['AssociationFeeFrequency']
        .astype('string')
        .str.strip()
        .str.lower()
        .map(HOA_PER_YEAR)
    )

    # Whether an HOA fee appears to exist
    # 0 = no HOA / $0 fee
    # 1 = HOA fee is reported
    # NaN = HOA information is missing
    out['has_hoa'] = np.where(
        fee.isna(),
        np.nan,
        (fee > 0).astype(float)
    )

    # Convert HOA fee to a monthly amount & missing/uninterpretable fees stay as NaN
    out['hoa_monthly'] = np.nan
    valid_fee = fee.notna() & (fee >= 0)

    out.loc[valid_fee & (fee == 0), 'hoa_monthly'] = 0

    # Positive fee and known frequency converted to monthly cost
    valid_frequency = valid_fee & (fee > 0) & mult.notna()
    out.loc[valid_frequency, 'hoa_monthly'] = (
        fee[valid_frequency] / mult[valid_frequency]
    )

    # Flag unrealistically large HOA values as missing
    big = out['hoa_monthly'] > HOA_MONTHLY_MAX

    log['hoa_monthly set NaN for values too large'] = int(big.sum())
    out.loc[big, 'hoa_monthly'] = np.nan

    # 6. YearBuilt
    sale_year = pd.to_datetime(out['CloseDate']).dt.year
    future = out['YearBuilt'] > sale_year
    out.loc[future, 'YearBuilt'] = sale_year[future]
    old = out['YearBuilt'] < YEAR_MIN
    out.loc[old, 'YearBuilt'] = np.nan
    log['YearBuilt after sale year -> sale year'] = int(future.sum())
    log[f'YearBuilt < {YEAR_MIN} -> NaN'] = int(old.sum())

    # 7. Drop raw columns now replaced plus the sparse ones
    out = out.drop(columns=['Levels', 'Flooring', 'AssociationFeeFrequency', 'AssociationFee'] + SPARSE_DROP,
                   errors='ignore')
    return out, log


before_fixed = clean_df.copy()
clean_df, log = fixed_cleaning(before_fixed)

print(pd.Series(log).to_string())
print(clean_df.shape)

MLSAreaMajor junk -> NaN                    36823
HighSchoolDistrict 'Other' -> NaN           26370
rows with more than one level value          2184
hoa_monthly set NaN for values too large      111
YearBuilt after sale year -> sale year         29
YearBuilt < 1790 -> NaN                         1
(358166, 71)


In [37]:
# Add YYYY-MM column
clean_df['file_month'] = (
    pd.to_datetime(clean_df['source_file'].str.extract(r'(\d{6})')[0], format='%Y%m')
      .dt.to_period('M')
)
print(clean_df['file_month'].value_counts().sort_index().to_string())

# Features
level_cols = [f'levels_{t}' for t in LEVEL_TOKENS] + ['levels_unknown']
floor_cols = [f'floor_{m}' for m in FLOOR_TYPES] + ['floor_unknown']

numeric_feats = [c for c in [
    'hoa_monthly', 'BathroomsTotalInteger', 'BedroomsTotal', 'GarageSpaces',
    'Latitude', 'LivingArea', 'Longitude', 'LotSizeSquareFeet', 'MainLevelBedrooms',
    'ParkingTotal', 'Stories', 'YearBuilt'] if c in clean_df.columns]

yn_feats   = binary_cols + ['has_hoa']      # 1/0 w/ NaN
flag_feats = level_cols + floor_cols        # 1/0 w/o NaN

feature_cols = numeric_feats + yn_feats + flag_feats + te_cols
clean_df[yn_feats] = clean_df[yn_feats].astype(float)

file_month
2024-01     8319
2024-02     9564
2024-03    11632
2024-04    12741
2024-05    13828
2024-06    12542
2024-07    13376
2024-08    12410
2024-09    10896
2024-10    12345
2024-11    10690
2024-12    10613
2025-01     8138
2025-02     8844
2025-03    10607
2025-04    11875
2025-05    11768
2025-06    11696
2025-07    12110
2025-08    11447
2025-09    11451
2025-10    12023
2025-11     9734
2025-12    10453
2026-01     7487
2026-02     8545
2026-03    11174
2026-04    12023
2026-05    12021
2026-06    13505
2026-07    12866
2026-08    11443
Freq: M


In [38]:
out_dir = 'data/sold/cleaned'
os.makedirs(out_dir, exist_ok=True)

# One cleaned file per original CSV
for src, part in clean_df.groupby('source_file'):
    stem, ext = os.path.splitext(os.path.basename(src))
    name = f'{stem}_cleaned{ext}'
    part.drop(columns='source_file').to_csv(os.path.join(out_dir, name), index=False)
    print(f'{name:40} {len(part):>8,} rows')

# One combined file
combined_path = 'data/sold/cleaned_all.csv'
clean_df.drop(columns='source_file').to_csv(combined_path, index=False)

print(f'\nTotal: {len(clean_df):,} rows across {clean_df["source_file"].nunique()} files')
print(f'Combined file: {combined_path}')

CRMLSSold202401_filled_cleaned.csv          8,319 rows
CRMLSSold202402_filled_cleaned.csv          9,564 rows
CRMLSSold202403_filled_cleaned.csv         11,632 rows
CRMLSSold202404_filled_cleaned.csv         12,741 rows
CRMLSSold202405_filled_cleaned.csv         13,828 rows
CRMLSSold202406_filled_cleaned.csv         12,542 rows
CRMLSSold202407_filled_cleaned.csv         13,376 rows
CRMLSSold202408_cleaned.csv                12,410 rows
CRMLSSold202409_cleaned.csv                10,896 rows
CRMLSSold202410_cleaned.csv                12,345 rows
CRMLSSold202411_cleaned.csv                10,690 rows
CRMLSSold202412_cleaned.csv                10,613 rows
CRMLSSold202501_filled_cleaned.csv          8,138 rows
CRMLSSold202502_cleaned.csv                 8,844 rows
CRMLSSold202503_cleaned.csv                10,607 rows
CRMLSSold202504_cleaned.csv                11,875 rows
CRMLSSold202505_cleaned.csv                11,768 rows
CRMLSSold202506_cleaned.csv                11,696 rows
CRMLSSold2

In [39]:
keep = feature_cols + [target, 'file_month']
clean_df[keep].to_csv('data/sold/cleaned_model_input.csv', index=False)

Missing values: Impossible values are set to NaN and rows with unusable prices are dropped. \
Missing HOA information set as NaN & valid HOA fees are converted to monthly amounts. \
Remaining NaNs are imputed with the training-window median (with a "missing indicator" column). \
Yes/No fields default to 0. \
Missing categorical values become their own "missing" level. \
Imputation, encoding, and scaling are fit only on the training months only to avoid leakage, so the cleaned CSVs can still contain NaNs.

In [40]:
# Preprocessing on training data
def build_preprocessor():
    return ColumnTransformer(
        [
            ('num', Pipeline([('imp', SimpleImputer(strategy='median', add_indicator=True)),
                              ('scale', StandardScaler())]), numeric_feats),
            ('yn', SimpleImputer(strategy='constant', fill_value=0, add_indicator=True), yn_feats),
            ('flags', 'passthrough', flag_feats),
            ('te', Pipeline([('enc', TargetEncoder(target_type='continuous',
                                       cv=KFold(n_splits=5, shuffle=True, random_state=0))),
                 ('scale', StandardScaler())]), te_cols),
        ],
        verbose_feature_names_out=False,
    ).set_output(transform='pandas')

# August 2026 = test ----------- previous n months = train
TEST_MONTH = pd.Period('2026-08', freq='M')

def preprocess_split(df, n_train_months, test_month=TEST_MONTH):
    train_months = pd.period_range(end=test_month - 1, periods=n_train_months, freq='M')

    # Check if data exists
    available = set(df['file_month'].unique())
    if test_month not in available:
        raise ValueError(f'No data for test month {test_month}')
    missing = [str(m) for m in train_months if m not in available]
    if missing:
        raise ValueError(f'No data for training months: {missing}. Try a smaller n_train_months.')

    train = df[df['file_month'].isin(train_months)]
    test = df[df['file_month'] == test_month]
    print(f'Train: {train_months[0]} to {train_months[-1]} ({len(train):,} rows) | '
          f'Test: {test_month} ({len(test):,} rows)')

    # Fit using training data only
    prep = build_preprocessor()
    X_train = prep.fit_transform(train[feature_cols], train[target])
    X_test = prep.transform(test[feature_cols])
    return X_train, train[target], X_test, test[target], prep

In [41]:
def save_split(n_train_months, test_month=TEST_MONTH, base_dir='data/sold/split'):
    X_train, y_train, X_test, y_test, prep = preprocess_split(clean_df, n_train_months, test_month)

    out_dir = os.path.join(base_dir, f'{n_train_months}mo_test_{test_month}')
    os.makedirs(out_dir, exist_ok=True)

    train_path = os.path.join(out_dir, 'train.csv')
    test_path  = os.path.join(out_dir, 'test.csv')

    X_train.assign(**{target: y_train.values}).to_csv(train_path, index=False)
    X_test.assign(**{target: y_test.values}).to_csv(test_path, index=False)

    print(f'Saved {train_path}  {X_train.shape}')
    print(f'Saved {test_path}  {X_test.shape}')

save_split(n_train_months=6)

Train: 2026-02 to 2026-07 (70,134 rows) | Test: 2026-08 (11,443 rows)
Saved data/sold/split\6mo_test_2026-08\train.csv  (70134, 54)
Saved data/sold/split\6mo_test_2026-08\test.csv  (11443, 54)
